#### **NEM Review contract co-design workshop**

# Contract performance modelling: simulations

## Prepare environment and data

In [5]:
# Data handling
import geopandas as gpd
import numpy as np
import os
import pandas as pd
import pyarrow.compute as pc

# Visualisation
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns

# Custom functions
from scripts.common_functions import python_setup, get_mms_data, save_figure

# Set up Python
working_dir, charts_dir, data_dir = python_setup(2)

# Paths and variables
gen_info_path = os.path.join(data_dir, "gen_info.csv")
duid_data_path = os.path.join(data_dir, "duid_data.parquet")
market_data_path = os.path.join(data_dir, "market_data.parquet")
regions = ["NSW", "QLD", "SA", "VIC"]

# Load gen info
gen_info = pd.read_csv(
    gen_info_path,
    index_col="DUID")

## Finance and cost assumptions

Financing assumptions are variables, which can be adjusted. Cost assumptions are sourced from [CSIRO's GenCost 2024-25](https://www.csiro.au/en/research/technology-space/energy/Electricity-transition/GenCost) (using 2024 prices under the 'current policies' scenario).

#### Finance

| Metric | Value |
|-|-|
| Leverage ratio | 65% |
| Loan rate | 5% |
| Equity rate | 15% |
| Weighted average cost of capital | 8.5% |

#### Costs

| | Wind | Solar |
|-|-|-|
| Capital expenditure *(\$/kW)* | $3,351 | $1,463 |
| Economic life *(years)* | 25 | 30 |
| Fixed operating & maintenance costs *(\$/kW/year)* | $28 | $12 |
| Annualised costs ($/MW/year) | $355,432 | $148,133 |

In [6]:
# Costs and finance variables
costs = {
    "Wind": {
        "Capex": 3351,
        "Economic life": 25,
        "FOM": 28},
    "Solar": {
        "Capex": 1463,
        "Economic life": 30,
        "FOM": 12,},
    "Financing": {
        "Loan": 0.65,
        "Equity": 0.35,
        "Loan rate": 0.05,
        "Equity rate": 0.15,
        "WACC": 0.085}}

for tech in ["Wind", "Solar"]:    
    capex = costs[tech]["Capex"]
    life = costs[tech]["Economic life"]
    fom = costs[tech]["FOM"]
    wacc = costs["Financing"]["WACC"]
    crf = wacc * (1 + wacc) ** life / ((1 + wacc) ** life - 1)
    annual_capex = capex * crf
    total_annual_cost_mw = (annual_capex + fom) * 1000
    costs[tech]["Annualised costs per MW"] = total_annual_cost_mw

## Contract parameters

| Contract | Reference price | Percentage of plant contracted |
|-|-|-|
| Merchant (no contract) | spot price | 100% of generator's output |
| Run of plant PPA | $100/MWh| 80% of generator's output |
| Baseload swap | $100/MWh | 80% of generator's historic capacity factor |
| Ex-post DWA swap | $100/MWh | 80% of generator's historic capacity factor |
| Ex-post regional revenue swap | \$100/MWh $\times$ region's historic capacity factor | 80% of generator's historic capacity factor |

## Contract performance analysis

### Analysis functions

In [7]:
# Resampling period strings
def period_string(period):
    if period == "quarter":
        return "QE"
    elif period == "month":
        return "ME"
    else:
        return "W"

# Filter DUIDs
def filter_duids(region, tech):
    duids = gen_info[
        (gen_info["Region"] == region) &
        (gen_info["Technology"] == tech) &
        (gen_info["Late start"] == False)].index.tolist()
    return duids

# Load generation data
def load_gen_data(duids):
    gen_data = pd.read_parquet(
        duid_data_path,
        engine="pyarrow",filters=[("DUID", "in", duids)],
        columns=["Interval", "DUID", "Output", "Maximum capacity"])
    return gen_data

# Rescale output
def rescale_output(gen_data):
    gen_data["Output"] = gen_data["Output"] / gen_data["Maximum capacity"] / 12
    gen_data.drop(columns="Maximum capacity", inplace=True)
    return gen_data

# Add price data
def add_price_data(gen_data, region):
    price_data = pd.read_parquet(
        market_data_path,
        engine="pyarrow",
        filters=[("Region", "==", region)],
        columns=["Interval", "Price"]).set_index("Interval")
    gen_data["Price"] = price_data.loc[gen_data["Interval"], "Price"].values
    del price_data
    return gen_data

# Add spot revenue
def add_spot_revenue(gen_data):
    gen_data["Spot revenue"] = (gen_data["Output"] * gen_data["Price"]).values
    return gen_data

# Contract output
def add_contract_output(contract, gen_data, plant_coverage, cfs=None):
    match contract:
        case "PPA":
            gen_data["Contract output"] = gen_data["Output"] * plant_coverage
        case "Baseload":
            # print(cfs)
            pass
            # gen_data["Contract output"] = gen_info.loc[gen_data["DUID"], "Capacity factor"] * plant_coverage
    return gen_data

# Collect capacity factors
def collect_capacity_factors(gen_data):
    duid_list = gen_data["DUID"]
    cfs = gen_info.loc[duid_list, "Capacity factor"]
    print(cfs)
    return cfs

# Contract revenue
def add_contract_revenue(gen_data, reference_price):
    gen_data["Contract revenue"] = gen_data["Contract output"] * reference_price
    return gen_data

# Payments to buyers
def add_payments(contract, gen_data, plant_coverage):
    match contract:
        case "PPA":
            gen_data["Payments"] = gen_data["Spot revenue"] * plant_coverage
    return gen_data

# Aggregate/resample data
def aggregate_gen_data(contract, gen_data, period):
    match contract:
        case "Spot price":
            values = "Spot revenue"
        case "PPA":
            values = ["Spot revenue", "Contract revenue", "Payments"]
    gen_data = gen_data.pivot(
        index="Interval",
        columns="DUID",
        values=values)
    revenue = gen_data.resample(period_string(period)).sum()
    # Combine revenues
    match contract:
        case "PPA":
            revenue = revenue["Spot revenue"] + revenue["Contract revenue"] - revenue["Payments"]
        case _:
            pass
    return revenue

# Substract average costs
def subtract_costs(revenue, costs, tech, period):
    earnings = revenue - costs[tech]["Annualised costs per MW"] / (4 if period == "quarter" else 12 if period == "month" else 52)
    return earnings

# Reformat earnings for merging
def reformat_earnings(contract, earnings, region, tech):
    earnings = earnings.reset_index().melt(
        id_vars="Interval",
        var_name="DUID",
        value_name="Earnings").rename(columns={"Interval": "Period"})
    earnings["Region"] = region
    earnings["Technology"] = tech
    match contract:
        case "Spot price":
            earnings["Contract"] = "Spot price (merchant)"
        case "PPA":
            earnings["Contract"] = "Run of plant PPA"
        case "Baseload":
            earnings["Contract"] = "Baseload swaps"
        case _:
            earnings["Contract"] = "Other"
    return earnings

# Contract performance analysis
def earnings_analysis(contract, region, tech="Wind", period="quarter", reference_price=100, plant_coverage=0.8):
    print(f"{contract} analysis for", region, tech.lower(), "...")

    # Standard functions
    duids = filter_duids(region, tech)
    gen_data = load_gen_data(duids)
    gen_data = rescale_output(gen_data)
    gen_data = add_price_data(gen_data, region)
    gen_data = add_spot_revenue(gen_data)

    # Contract functions
    match contract:
        case "Spot price":
            pass
        case "PPA":
            gen_data = add_contract_output(contract, gen_data, plant_coverage)
            # gen_data = add_contract_revenue(gen_data, reference_price)
            # gen_data = add_payments(contract, gen_data, plant_coverage)
        case "Baseload":
            cfs = collect_capacity_factors(gen_data)            
            gen_data = add_contract_output(contract, gen_data, plant_coverage, cfs)
        #     gen_data = add_contract_revenue(gen_data, reference_price)
        #     gen_data = add_payments(contract, gen_data, plant_coverage)
        case _:
            pass

    # Prepare results
    # revenue = aggregate_gen_data(contract, gen_data, period)
    # del gen_data
    # earnings = subtract_costs(revenue, costs, tech, period)
    # del revenue
    # earnings = reformat_earnings(contract, earnings, region, tech)

    # Return results
    # return earnings
    return gen_data

In [ ]:
tests = earnings_analysis("Baseload", "SA")
tests

### Perform analysis

In [ ]:
# Dataframe for results
station_earnings = pd.DataFrame(columns=["Contract", "Region", "Technology", "DUID", "Period", "Earnings"])

# Add earnings
for region in regions[2:3]:
    for tech in ["Wind"]:
        # Spot market
        merchant_earnings = earnings_analysis("Spot price", region, tech)
        # Run of plant PPA
        ppa_earnings = earnings_analysis("PPA", region, tech)
        # Run of plant PPA
        baseload_earnings = earnings_analysis("Baseload", region, tech)
        if station_earnings.empty:
            station_earnings = pd.concat([merchant_earnings, ppa_earnings])
        else:
            station_earnings = pd.concat([station_earnings, merchant_earnings, ppa_earnings])

# Calculate variances
station_stds = station_earnings.pivot_table(
    index=["DUID", "Region", "Technology"],
    columns="Contract",
    values="Earnings",
    aggfunc="std")

In [ ]:
chart_data = station_stds.copy().reset_index().melt(
    id_vars=["DUID", "Region", "Technology"],
    var_name="Contract",
    value_name="Standard deviation")
fig, ax = plt.subplots(
    figsize=(16, 9),
    tight_layout=True)
sns.boxplot(
    ax=ax,
    data=chart_data,
    x="Standard deviation",
    y="Contract",
    hue="Contract",
    orient="h",
    linewidth=3,
    fliersize=15,
    flierprops=dict({
        "markeredgewidth":3,
    }))
for frame in ["top", "right", "bottom", "left"]:
    ax.spines[frame].set_visible(False)
ax.xaxis.set_major_formatter(lambda x, p: f"${x / 1000:,.0f}k")
plt.title("SA wind contract simulations (16 generators)", fontsize="large")
plt.ylabel(None)
plt.xlabel(f"Standard deviation of earnings per quarter")
plt.show()

In [ ]:
chart_data = station_earnings.copy()
fig, ax = plt.subplots(
    figsize=(16, 9),
    tight_layout=True)
sns.boxplot(
    ax=ax,
    data=chart_data,
    x="Period",
    y="Earnings",
    hue="Contract",
    # linewidth=2,
    fliersize=10,
    # flierprops=dict({
    #     "markeredgewidth":2,
    # }))
)
plt.axhline(
    y=0,
    color="C1",
    linestyle="--")
for frame in ["top", "right", "bottom", "left"]:
    ax.spines[frame].set_visible(False)
ax.set_xticks(ax.get_xticks()[::3])
ax.set_xticklabels([f"{chart_data['Period'].sort_values().unique()[t]:%b %Y}" for t in ax.get_xticks()])
ax.yaxis.set_major_formatter(lambda x, p: f"${x / 1000:,.0f}k")
plt.title("SA wind contract simulations (16 generators)", fontsize="large")
plt.legend(title=None)
plt.xlabel(None)
plt.ylabel("Earnings per quarter")
plt.show()

In [ ]:
# Dataframe for results
simulated_stations = pd.DataFrame(
    index=duids)
simulated_stations.index.name = "DUID"
simulated_stations["Technology"] = tech
simulated_stations["Region"] = region
simulated_stations["Name"] = gen_info.loc[filtered_duids, "Name"]
simulated_stations["Capacity factor"] = gen_info.loc[filtered_duids, "Capacity factor"]
simulated_stations.sort_values("Capacity factor", ascending=False, inplace=True)
    
    
    # Contract-specific calculations
    filtered_gen_data["Plant contract output"] = (gen_info.loc[filtered_gen_data["DUID"], "Capacity factor"] * plant_coverage).values
    filtered_gen_data["PPA contract revenue"] = filtered_gen_data["Output"] * plant_coverage * ppa_price / 12
    filtered_gen_data["PPA spot revenue"] = filtered_gen_data["Output"] * filtered_gen_data["Price"] * (1 - plant_coverage) / 12 
    filtered_gen_data["Baseload contract revenue"] = (filtered_gen_data["Plant contract output"] * baseload_price).values / 12
    filtered_gen_data["Baseload spot revenue"] = ((filtered_gen_data["Output"] - filtered_gen_data["Plant contract output"]) * filtered_gen_data["Price"]).values / 12
    filtered_gen_data["DWA swap region profile"] = region_gen_profile.loc[filtered_gen_data["Interval"], "Output"].values        
    filtered_gen_data["DWA swap contract revenue"] = (filtered_gen_data["Plant contract output"] * dwa_swap_price).values / 12
    filtered_gen_data["DWA swap spot revenue"] = ((filtered_gen_data["Output"] - filtered_gen_data["Plant contract output"]) * filtered_gen_data["Price"]).values / 12
    regional_revenue_swap_capacity_factor = region_gen_profile["Output"].mean()
    regional_revenue_swap_revenue_per_mw_per_interval = regional_revenue_swap_capacity_factor * regional_revenue_swap_price_unadjusted / 12        
    filtered_gen_data["Regional revenue swap contract revenue"] = (filtered_gen_data["Plant contract output"] * regional_revenue_swap_revenue_per_mw_per_interval).values
    filtered_gen_data["Regional revenue swap spot revenue"] = ((filtered_gen_data["Output"] - filtered_gen_data["Plant contract output"]) * filtered_gen_data["Price"]).values / 12
    
    # Prepare PPA data
    ppa_data = filtered_gen_data.pivot_table(
        index="Interval",
        columns="DUID",
        values=["PPA contract revenue", "PPA spot revenue"],
        aggfunc="sum")
    # Prepare baseloard data
    baseload_data = filtered_gen_data.pivot_table(
        index="Interval",
        columns="DUID",
        values=["Baseload contract revenue", "Baseload spot revenue"],
        aggfunc="sum")
    # Prepare DWA swap data
    dwa_swap_data = filtered_gen_data.pivot_table(
        index="Interval",
        columns="DUID",
        values=["DWA swap contract revenue", "DWA swap spot revenue"],
        aggfunc="sum")
    # Prepare Regional revenue swap data
    regional_revenue_swap_data = filtered_gen_data.pivot_table(
        index="Interval",
        columns="DUID",
        values=["Regional revenue swap contract revenue", "Regional revenue swap spot revenue"],
        aggfunc="sum")
    del filtered_gen_data

    # Aggregate to months/quarters
    ppa_data_mth = ppa_data.resample("ME").sum()
    ppa_data_mth = ppa_data_mth["PPA contract revenue"] + ppa_data_mth["PPA spot revenue"]
    ppa_data_qtr = ppa_data.resample("QE").sum()
    ppa_data_qtr = ppa_data_qtr["PPA contract revenue"] + ppa_data_qtr["PPA spot revenue"]
    baseload_data_mth = baseload_data.resample("ME").sum()
    baseload_data_mth = baseload_data_mth["Baseload contract revenue"] + baseload_data_mth["Baseload spot revenue"]
    baseload_data_qtr = baseload_data.resample("QE").sum()
    baseload_data_qtr = baseload_data_qtr["Baseload contract revenue"] + baseload_data_qtr["Baseload spot revenue"]
    dwa_swap_data_mth = dwa_swap_data.resample("ME").sum()
    dwa_swap_data_mth = dwa_swap_data_mth["DWA swap contract revenue"] + dwa_swap_data_mth["DWA swap spot revenue"]
    dwa_swap_data_qtr = dwa_swap_data.resample("QE").sum()
    dwa_swap_data_qtr = dwa_swap_data_qtr["DWA swap contract revenue"] + dwa_swap_data_qtr["DWA swap spot revenue"]
    regional_revenue_swap_data_mth = regional_revenue_swap_data.resample("ME").sum()
    regional_revenue_swap_data_mth = regional_revenue_swap_data_mth["Regional revenue swap contract revenue"] + regional_revenue_swap_data_mth["Regional revenue swap spot revenue"]
    regional_revenue_swap_data_qtr = regional_revenue_swap_data.resample("QE").sum()
    regional_revenue_swap_data_qtr = regional_revenue_swap_data_qtr["Regional revenue swap contract revenue"] + regional_revenue_swap_data_qtr["Regional revenue swap spot revenue"]
    del ppa_data, baseload_data, dwa_swap_data, regional_revenue_swap_data
    # Substract average costs
    ppa_data_mth = ppa_data_mth - costs[tech]["Annualised costs per MW"] / 12
    baseload_data_mth = baseload_data_mth - costs[tech]["Annualised costs per MW"] / 12
    dwa_swap_data_mth = dwa_swap_data_mth - costs[tech]["Annualised costs per MW"] / 12
    regional_revenue_swap_data_mth = regional_revenue_swap_data_mth - costs[tech]["Annualised costs per MW"] / 12        
    ppa_data_qtr = ppa_data_qtr - costs[tech]["Annualised costs per MW"] / 4
    baseload_data_qtr = baseload_data_qtr - costs[tech]["Annualised costs per MW"] / 4
    dwa_swap_data_qtr = dwa_swap_data_qtr - costs[tech]["Annualised costs per MW"] / 4
    regional_revenue_swap_data_qtr = regional_revenue_swap_data_qtr - costs[tech]["Annualised costs per MW"] / 4
    # Attach earnings/MW standard deviation
    simulated_stations["Run of plant PPA (mth)"] = ppa_data_mth.std()
    simulated_stations["Baseload swap (mth)"] = baseload_data_mth.std()
    simulated_stations["Ex-post DWA swap (mth)"] = dwa_swap_data_mth.std()        
    simulated_stations["Regional revenue swap (mth)"] = regional_revenue_swap_data_mth.std()
    simulated_stations["Run of plant PPA (qtr)"] = ppa_data_qtr.std()
    simulated_stations["Baseload swap (qtr)"] = baseload_data_qtr.std()
    simulated_stations["Ex-post DWA swap (qtr)"] = dwa_swap_data_qtr.std()
    simulated_stations["Regional revenue swap (qtr)"] = regional_revenue_swap_data_qtr.std()